# Data Transformation

In [1]:
1 +1 

2

In [2]:
import os
%pwd

'd:\\MLOps Udemy Krish Naik\\Text-Summarizer\\research'

In [3]:
os.chdir("../")
%pwd

'd:\\MLOps Udemy Krish Naik\\Text-Summarizer'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataTransformationConfig:
    root_dir:Path
    data_path: Path
    tokenizer_name: Path

In [7]:
from src.text_summarizer.constants import *
from src.text_summarizer.utils.common import read_yaml,create_dir
from src.text_summarizer.exception.exception import SummaryException
import sys

class ConfigurationManager:
    def __init__(self,config_path=CONFIG_FILE_PATH,params_file_path=PARAMS_FILE_PATH):
        self.config = read_yaml(path_to_yaml=config_path)
        self.params = read_yaml(path_to_yaml=params_file_path)

        create_dir([self.config.artifacts_root])

    def get_data_transformation_config(self)->DataTransformationConfig:
        try:
            config = self.config.data_transformation

            create_dir([config.root_dir])

            data_transformation_config = DataTransformationConfig(
                root_dir=config.root_dir,
                data_path=config.data_path,
                tokenizer_name= config.tokenizer_name
            )
            return data_transformation_config
        except Exception as e:
            raise SummaryException(e,sys)

In [8]:
import os
from src.text_summarizer.logging_info import logger
from transformers import AutoTokenizer
from datasets import load_from_disk

### Data Transformation Components

In [9]:
class DataTransformation:
    def __init__(self,config:DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name) 
    
    def convert_to_features(self,example_batch):
        try:
            input_encodings = self.tokenizer(example_batch['dialogue'] , max_length = 1024, truncation = True )

            with self.tokenizer.as_target_tokenizer():
                target_encodings = self.tokenizer(example_batch['summary'], max_length = 128, truncation = True )

            return {
                'input_ids' : input_encodings['input_ids'],
                'attention_mask': input_encodings['attention_mask'],
                'labels': target_encodings['input_ids']
            }
        except Exception as e:
            raise SummaryException(e,sys)
        
    def convert(self):
        try:
            data_sam = load_from_disk(self.config.data_path)
            data_sam_pt = data_sam.map(self.convert_to_features, batched = True)
            data_sam_pt.save_to_disk(os.path.join(self.config.root_dir,"samsung_dataset"))
        except Exception as e:
            raise SummaryException(e,sys)
        


In [11]:
config = ConfigurationManager()
data_transformation_config = config.get_data_transformation_config()
data_transformation = DataTransformation(config=data_transformation_config)
data_transformation.convert()

Map:   0%|          | 0/14732 [00:00<?, ? examples/s]d:\MLOps Udemy Krish Naik\Text-Summarizer\venv\lib\site-packages\transformers\tokenization_utils_base.py:3980: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 110972.63 examples/s]
